# Sprint 4 — Kaggle CPU Inference

Notebook de soumission BirdCLEF 2026.

- **CPU only** (regle competition : GPU submissions = 1 min max)
- **Internet OFF** — toutes les dependances doivent etre pre-attachees
- **Output** : `submission.csv`

Prerequis datasets a attacher :
1. **Competition** `birdclef-2026`
2. **Code V2** `hellodave2035/birdclef-v2-sprint4-code`
3. **Checkpoints** `hellodave2035/birdclef-v2-sprint4-checkpoints`

Attention : `test_soundscapes/` est **vide en dry-run**. Normal.
Les `.ogg` apparaissent uniquement lors du scoring cache.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, time
import pandas as pd

NOTEBOOK_VERSION = 'sprint4-v1'
INPUT_ROOT = Path('/kaggle/input')
COMP_ROOT = INPUT_ROOT / 'competitions' / 'birdclef-2026'
WORK_ROOT = Path('/kaggle/working/__birdclef-V2')

print(f'Notebook: {NOTEBOOK_VERSION}')
print(f'Python: {sys.version.split()[0]}')
print(f'sample_submission.csv: {(COMP_ROOT / "sample_submission.csv").exists()}')
print(f'test_soundscapes: {(COMP_ROOT / "test_soundscapes").exists()}')

assert COMP_ROOT.exists(), 'Missing competition dataset'
assert (COMP_ROOT / 'sample_submission.csv').exists(), 'sample_submission.csv not found'

In [ ]:
# === Find and copy the V2 repo from attached datasets ===
REPO_DATASET_NESTED_ROOTS = (
    '__birdclef-V2',
    'hellodave2035/birdclef-v2-sprint4-code',
    'hellodave2035/birdclef-v2-kaggle-train-full-weights-2026-04-30e',
    'hellodave2035/birdclef-v2-kaggle-train-repo-2026-04-30d',
    'hellodave2035/birdclef-v2-kaggle-train-repo',
)

def find_repo_source(input_root):
    candidates = []
    for dataset_dir in sorted(p for p in input_root.iterdir() if p.is_dir()):
        roots = [dataset_dir] + [dataset_dir / r for r in REPO_DATASET_NESTED_ROOTS]
        for root in roots:
            if not root.exists():
                continue
            if (root / 'run_context.json').exists() and (root / 'scripts' / 'sprint4_inference.py').exists():
                candidates.append(root.resolve())
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError('No V2 repo found. Attach birdclef-v2-sprint4-code dataset.')
    if len(candidates) > 1:
        print('Multiple repos:', candidates)
        raise RuntimeError('Ambiguous. Keep only one code dataset.')
    return candidates[0]

REPO_SOURCE = find_repo_source(INPUT_ROOT)
print('Repo source:', REPO_SOURCE)

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
shutil.copytree(REPO_SOURCE, WORK_ROOT,
    ignore=shutil.ignore_patterns('.git', '.venv', '__pycache__', '.pytest_cache', '*.pyc', '*.pyo'))

os.chdir(WORK_ROOT)
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
print('Writable repo ready:', WORK_ROOT)

In [ ]:
# === Install inference-only deps ===
MISSING = []
for pkg in ['timm', 'soundfile']:
    try:
        __import__(pkg)
        print(f'{pkg}: OK')
    except ImportError:
        MISSING.append(pkg)

if MISSING:
    for pkg in MISSING:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
    for pkg in MISSING:
        __import__(pkg)
        print(f'{pkg}: installed')

import torch, timm, soundfile
print(f'Torch: {torch.__version__}  |  Timm: {timm.__version__}  |  SoundFile: {soundfile.__version__}')

In [ ]:
# === Find checkpoints ===
def find_checkpoints_dir(input_root):
    candidates = []
    for dataset_dir in sorted(input_root.iterdir()):
        if not dataset_dir.is_dir() or dataset_dir.name == 'competitions':
            continue
        for match in dataset_dir.glob('**/fold_0.pth'):
            candidates.append(match.parent)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError('No fold_*.pth found. Attach checkpoints dataset.')
    if len(candidates) > 1:
        print('Multiple:', candidates)
        raise RuntimeError('Ambiguous. Keep only one checkpoints dataset.')
    return candidates[0]

CKPT_DIR = find_checkpoints_dir(INPUT_ROOT)
print('Checkpoints dir:', CKPT_DIR)
for f in sorted(CKPT_DIR.glob('fold_*.pth')):
    print(f'  {f.name}: {f.stat().st_size / 1e6:.1f} MB')

In [ ]:
# === Run inference ===
TEST_DIR = COMP_ROOT / 'test_soundscapes'
SAMPLE_SUB = COMP_ROOT / 'sample_submission.csv'
SUBM_OUT = Path('/kaggle/working/submission.csv')

print(f'Test dir: {TEST_DIR}')
ogg_files = sorted(TEST_DIR.glob('*.ogg')) if TEST_DIR.exists() else []
print(f'.ogg files: {len(ogg_files)}')
if ogg_files:
    print(f'  First: {ogg_files[0].name}')

script = WORK_ROOT / 'scripts' / 'sprint4_inference.py'
assert script.exists(), f'Missing: {script}'

t0 = time.time()
result = subprocess.run([
    sys.executable, str(script),
    '--soundscapes-dir', str(TEST_DIR),
    '--checkpoints-dir', str(CKPT_DIR),
    '--sample-submission', str(SAMPLE_SUB),
    '--output', str(SUBM_OUT),
    '--batch-size', '32',
], cwd=WORK_ROOT)

elapsed = time.time() - t0
print(f'Inference done in {elapsed:.0f}s ({elapsed/60:.1f} min)')
if result.returncode != 0:
    raise RuntimeError(f'Inference failed (rc={result.returncode})')

In [ ]:
# === Validate ===
sample = pd.read_csv(SAMPLE_SUB)
sub = pd.read_csv(SUBM_OUT)

print(f'Sample: {len(sample)} rows x {len(sample.columns)} cols')
print(f'Submission: {len(sub)} rows x {len(sub.columns)} cols')
print(f'Shape match: {sub.shape == sample.shape}')
print(f'Columns match: {list(sub.columns) == list(sample.columns)}')
print(f'Row IDs match: {sub["row_id"].equals(sample["row_id"])}')

proba = sub.drop(columns=['row_id'])
print(f'NaN: {proba.isna().sum().sum()}')
print(f'Range: [{proba.min().min():.6f}, {proba.max().max():.6f}]')
print(f'Mean: {proba.mean().mean():.6f}')

print(f'\nSubmission ready: {SUBM_OUT}')
print(f'Size: {SUBM_OUT.stat().st_size / 1e6:.2f} MB')

## Notes

- **Dry-run** : `test_soundscapes/` vide → fallback `sample_submission.csv` → score 0.500
- **Scoring reel** : `.ogg` presents → inference reelle → score > 0.500
- **Row IDs** : `BC2026_Test_XXXX_..._5`, `_10`, `_15` (end time en secondes)
- **90 min CPU** : ~400 fichiers x 5s x 5 folds ≈ 2.8h → optimiser si timeout (batch_size, reduction folds)